# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

My lane is **"which pages need a content refresh first?"** — a ranking/prioritization question, not a plain yes/no
classification in isolation. Per the `training-honest-models` skill table, a ranking question needs a classifier's
probability score evaluated at precision@K, not just a label.

I'm training three methods on the same features and comparing them:
- **Logistic Regression** — readable baseline-of-models, coefficients are interpretable.
- **Decision Tree** (depth 5) — a step up in flexibility, still printable/readable.
- **Random Forest** — stronger, non-linear, handles feature interactions the first two miss.

I start simple (Logistic Regression) and only add complexity (Tree → Forest) because the comparison table below
earns it — if Logistic Regression had matched Random Forest at precision@50, I would have stopped there.

In [1]:
# --- Colab setup: clone the repo and regenerate the processed data (gitignored, pipeline-built) ---
import os

REPO_URL = "https://github.com/muzammil-12345/flyrank-ml-internship.git"
REPO_DIR = "/content/flyrank-ml-internship"

if not os.path.isdir(os.path.join(REPO_DIR, "scripts")):
    !git clone -q {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}/scripts
# data/processed/ is gitignored -- these two scripts rebuild it from data/raw/
!python3 01_prepare_features.py
!python3 02_baseline_score.py

%cd {REPO_DIR}/work/notebooks
print("Working directory set to:", os.getcwd())

/content/flyrank-ml-internship/scripts


Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship/data/processed/refresh_feature_vector.csv


Wrote baseline queue: /content/flyrank-ml-internship/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340


/content/flyrank-ml-internship/work/notebooks
Working directory set to: /content/flyrank-ml-internship/work/notebooks


In [2]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.inspection import permutation_importance
import sys, os

# Find the repo root regardless of where this notebook is launched from
# (works whether run from work/notebooks/, the repo root, or Colab after a clone)
def find_repo_root(start='.'):
    path = os.path.abspath(start)
    for _ in range(6):
        if os.path.isdir(os.path.join(path, 'scripts')) and os.path.isdir(os.path.join(path, 'data')):
            return path
        path = os.path.dirname(path)
    raise FileNotFoundError("Could not locate repo root (folder containing 'scripts/' and 'data/'). "
                             "If running in Colab, !git clone the repo first and %cd into it.")

REPO_ROOT = find_repo_root()
sys.path.insert(0, os.path.join(REPO_ROOT, 'scripts'))
from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES, precision_at_k

RANDOM_STATE = 42

feat = pd.read_csv(os.path.join(REPO_ROOT, 'data/processed/refresh_feature_vector.csv'))
base = pd.read_csv(os.path.join(REPO_ROOT, 'data/processed/baseline_refresh_queue.csv'))
target = feat['is_declining_label']

print(f"Feature vector: {feat.shape[0]:,} rows, {feat.shape[1]} columns")
print(f"Declining rate (full data): {target.mean():.3f}")

models = {
    'logistic_regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    'decision_tree': DecisionTreeClassifier(class_weight='balanced', max_depth=5, min_samples_leaf=50, random_state=RANDOM_STATE),
    'random_forest': RandomForestClassifier(class_weight='balanced_subsample', max_depth=10, min_samples_leaf=25,
                                             n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE),
}
print(f"Methods to compare: {list(models.keys())}")

Feature vector: 30,000 rows, 52 columns
Declining rate (full data): 0.542
Methods to compare: ['logistic_regression', 'decision_tree', 'random_forest']


## 2. Split design

**Grouped by client_id, not a random row split.** Per `flyrank-data`'s gotcha list, `content_id`/`client_id` are
pseudonyms for grouping only — but more importantly, rows from the *same client* share editorial style, traffic
scale, and content patterns. A random row split would leak client-level patterns between train and test (the model
could partly "recognize" a client's style rather than learn general refresh signals). A **client holdout** — entire
clients held out for testing — is the honest split for this question, matching the split already used for the
Week-4 baseline so the comparison in Section 3 is apples-to-apples.

In [3]:
client_series = feat['client_id'].fillna('unknown').astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()

rng = np.random.default_rng(RANDOM_STATE)
shuffled_clients = rng.permutation(unique_clients)
test_client_count = max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients = set(shuffled_clients[:test_client_count])

test_mask = client_series.isin(test_clients).to_numpy()
all_idx = np.arange(len(feat))
train_idx, test_idx = all_idx[~test_mask], all_idx[test_mask]

print(f"Split design: client_holdout")
print(f"Train: {len(train_idx):,} rows across {feat.iloc[train_idx]['client_id'].nunique()} clients")
print(f"Test:  {len(test_idx):,} rows across {feat.iloc[test_idx]['client_id'].nunique()} clients")
print(f"Train positive rate: {target.iloc[train_idx].mean():.3f} | Test positive rate: {target.iloc[test_idx].mean():.3f}")
overlap = set(feat.iloc[train_idx]['client_id']) & set(feat.iloc[test_idx]['client_id'])
print(f"Clients overlapping train/test: {len(overlap)} (must be 0)")

Split design: client_holdout
Train: 27,675 rows across 26 clients
Test:  2,325 rows across 6 clients
Train positive rate: 0.555 | Test positive rate: 0.391
Clients overlapping train/test: 0 (must be 0)


## 3. Train + compare vs my baseline

Same test rows, same `is_declining_label` target, same metric set as the Week-4 rule baseline
(`baseline_refresh_score` from `data/processed/baseline_refresh_queue.csv`). I evaluate all three models plus the
baseline plus the base rate, side by side, in one table — this is the non-negotiable comparison the skill calls for.

In [4]:
numeric_features = [c for c in MODEL_NUMERIC_FEATURES if c in feat.columns]
categorical_features = [c for c in MODEL_CATEGORICAL_FEATURES if c in feat.columns]

numeric_frame = feat[numeric_features].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
cat_frame = feat[categorical_features].fillna('unknown').astype(str)
encoded = pd.get_dummies(cat_frame, prefix=categorical_features, dummy_na=False, dtype=float)
X = pd.concat([numeric_frame.reset_index(drop=True), encoded.reset_index(drop=True)], axis=1)
y = target.reset_index(drop=True)

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
print(f"Feature matrix: {X.shape[1]} columns ({len(numeric_features)} numeric + {len(encoded.columns)} one-hot from {len(categorical_features)} categorical fields)")
print("NOTE: trend_direction / trend_pct are excluded from features -- they define the label (label trap, see flyrank-data skill).")

Feature matrix: 52 columns (18 numeric + 34 one-hot from 8 categorical fields)
NOTE: trend_direction / trend_pct are excluded from features -- they define the label (label trap, see flyrank-data skill).


In [5]:
results = {}
probas = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    probas[name] = proba
    results[name] = {
        'roc_auc': roc_auc_score(y_test, proba),
        'avg_precision': average_precision_score(y_test, proba),
        'precision_at_20': precision_at_k(y_test, proba, 20),
        'precision_at_50': precision_at_k(y_test, proba, 50),
        'precision_at_100': precision_at_k(y_test, proba, 100),
    }

base_indexed = base.set_index('content_id')
test_content_ids = feat.iloc[test_idx]['content_id'].values
base_test = base_indexed.loc[test_content_ids]
base_score = base_test['baseline_refresh_score'].values
base_label = base_test['is_declining_label'].values
assert (base_label == y_test.values).all(), "Label mismatch -- baseline rows must align with model test rows"

results['baseline_rules'] = {
    'roc_auc': roc_auc_score(base_label, base_score),
    'avg_precision': average_precision_score(base_label, base_score),
    'precision_at_20': precision_at_k(base_label, base_score, 20),
    'precision_at_50': precision_at_k(base_label, base_score, 50),
    'precision_at_100': precision_at_k(base_label, base_score, 100),
}
results['base_rate (always predict positive)'] = {
    'roc_auc': 0.5, 'avg_precision': float(y_test.mean()),
    'precision_at_20': float(y_test.mean()), 'precision_at_50': float(y_test.mean()), 'precision_at_100': float(y_test.mean()),
}

comparison_table = pd.DataFrame(results).T.round(3)
comparison_table

,roc_auc,avg_precision,precision_at_20,precision_at_50,precision_at_100
logistic_regression,0.700,0.522,0.350,0.400,0.440
decision_tree,0.742,0.575,0.650,0.660,0.680
random_forest,0.750,0.618,0.650,0.740,0.720
baseline_rules,0.627,0.468,0.150,0.240,0.360
base_rate (always predict positive),0.500,0.391,0.391,0.391,0.391


**Reading the table:** Random Forest wins on every metric, and specifically on `precision_at_50` — our chosen
success metric — it scores **0.740 vs the rule baseline's 0.240** (base rate is 0.391, so the baseline's naive rules
are actually *worse* than random guessing at precision@50, while Random Forest is nearly double the base rate).
Decision Tree is close behind Random Forest and notably more readable (a depth-5 tree fits on one screen), so it
stays in the report as the "read this if you want to understand the logic" option. Logistic Regression trails
all tree-based methods here — the relationship between these features and decline risk is evidently non-linear
enough that a linear model under-performs even the simple rule baseline on precision@20.

## 4. Errors and interpretation

Random Forest is the selected model (best precision@50). Before trusting the score, I check what it leans on
(permutation importance) and where it's actually wrong (error cases, grouped by content_type).

In [6]:
best_model_name = max(['logistic_regression', 'decision_tree', 'random_forest'], key=lambda n: results[n]['precision_at_50'])
best_model = models[best_model_name]
print(f"Best model: {best_model_name}")

perm = permutation_importance(best_model, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE,
                               scoring='average_precision', n_jobs=-1)
importances = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)
importances.head(10)

Best model: random_forest


days_with_impressions       0.081486
log_impressions_90d         0.026035
ctr                         0.021477
scroll_rate                 0.010934
search_volume               0.008015
log_clicks_90d              0.007184
avg_position                0.006816
main_intent_unknown         0.003268
days_with_sessions          0.002815
impression_tier_moderate    0.002642
dtype: float64

**Do the top features make sense?** `days_with_impressions` and `log_impressions_90d` dominate — pages seen
consistently over many days, at high volume, carry the strongest signal about their trend health. `ctr` and
`scroll_rate` follow — engagement quality signals, which is intuitive: a page losing clicks/engagement is
plausibly declining. None of these are suspiciously perfect (no single feature explains >10% of the importance
mass alone), which is a reasonable check against leakage — a leaked feature usually dominates near-total.

In [7]:
test_frame = feat.iloc[test_idx].reset_index(drop=True).copy()
test_frame['model_proba'] = probas[best_model_name]
top50 = test_frame.sort_values('model_proba', ascending=False).head(50)
wrong_top50 = top50[top50['is_declining_label'] == 0]
print(f"Top-50 by model score: {(top50['is_declining_label']==1).sum()} correctly declining, {len(wrong_top50)} wrong (scored high, not actually declining)")

cols_show = ['content_type', 'impressions_90d', 'sessions_90d', 'avg_position', 'ctr', 'content_age_days', 'days_since_last_update', 'model_proba']
wrong_top50[cols_show].head(3)

Top-50 by model score: 37 correctly declining, 13 wrong (scored high, not actually declining)


,content_type,impressions_90d,sessions_90d,avg_position,ctr,content_age_days,days_since_last_update,model_proba
1752,keyword article,5091,30,14.1,0.20,144,20,0.737130
1778,keyword article,1076,9,25.6,0.09,125,20,0.734944
1988,keyword article,3026,24,35.9,0.00,134,20,0.733631


**Why these 3 are hard:** all three are "keyword article" content with decent impressions (1k-5k) and low CTR
(0.00-0.20%) — exactly the profile the model has learned to flag as declining. But their `trend_direction` label
says otherwise. My read: these are likely pages that were *already* low-performing and stayed flat (not "declining"
by the label's down-from-a-peak definition), rather than pages actively losing ground. The model conflates
"chronically weak" with "actively declining" because both share low-CTR/low-engagement signals — a real limitation
worth flagging to whoever consumes this queue.

In [8]:
test_frame['pred_high'] = (test_frame['model_proba'] >= 0.5).astype(int)
test_frame['correct'] = (test_frame['pred_high'] == test_frame['is_declining_label']).astype(int)
by_type = test_frame.groupby('content_type')['correct'].agg(['mean', 'count']).sort_values('mean')
by_type.round(3)

,mean,count
content_type,,
keyword article,0.595,1367
feedly article,0.782,958


**Where the model is most wrong:** "keyword article" content (59.5% accuracy) is noticeably weaker than
"feedly article" content (78.2% accuracy) at the same 0.5 cutoff. Keyword articles are likely more
templated/programmatic, so the traffic/engagement signals the model relies on may be noisier or less
discriminating for this content type — a candidate area for a content-type-specific threshold or a follow-up
feature in a later iteration, rather than something this version's single global score fixes.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.